# Технологии программирования, БИ

## НИУ ВШЭ, 2025-26 учебный год

# Семинар 3. ООП и тестирование

## Раздел 1. Объектно-ориентированное программирование в Python

**Цель:** понять, когда нужны классы, как устроены объекты в Python и как применять основные идеи ООП.

После семинара вы сможете:
1. создавать классы и объекты;
2. понимать роль `self` и `__init__`;
3. различать атрибуты экземпляра и атрибуты класса;
4. писать методы и магические методы `__str__`, `__repr__`, `__eq__`, `__add__`;
5. использовать инкапсуляцию через `_name` и `@property`;
6. применять наследование, `super()`, полиморфизм и ABC;
7. понимать, когда лучше использовать `dataclass`;

Материалы, на которые можно опираться после семинара:

- Хабр: [ООП в Python за 1 статью](https://habr.com/ru/articles/1000378/)
- Proglib: [ООП на Python: концепции, принципы и примеры реализации](https://proglib.io/p/python-oop/amp/)
- Яндекс Хэндбук: [Объектная модель Python](https://education.yandex.ru/handbook/python/article/obuektnaya-model-python-klassy-polya-i-metody)
- Документация Python: [Data model](https://docs.python.org/3/reference/datamodel.html), [dataclasses](https://docs.python.org/3/library/dataclasses.html), [abc](https://docs.python.org/3/library/abc.html)

In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from copy import copy, deepcopy
from dataclasses import dataclass, field
from math import pi

### Зачем нужно ООП

ООП удобно, когда в программе появляются сущности со своим состоянием и поведением.

Например, если мы пишем банковское приложение, у счёта есть:

- состояние: владелец, баланс;
- поведение: пополнить счёт, снять деньги, показать информацию.

Без класса такие данные быстро превращаются в набор разрозненных переменных и функций.

Класс в такой ситуации работает как аккуратная коробка: рядом лежат и данные, и операции над ними. Когда сущностей становится много, это сильно снижает хаос в коде.

In [ ]:
# Без ООП: данные и действия отделены друг от друга
account_owner = "Анна"
account_balance = 1000


def deposit(balance: int, amount: int) -> int:
    return balance + amount


account_balance = deposit(account_balance, 500)
print(account_owner, account_balance)

Анна 1500


In [ ]:
# С ООП: данные и действия живут вместе
class BankAccount:
    def __init__(self, owner: str, balance: int) -> None:
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: int) -> None:
        self.balance += amount


account = BankAccount("Анна", 1000)
account.deposit(500)
print(account.owner, account.balance)

Анна 1500


### Четыре базовые идеи ООП

| Идея | Простыми словами | Где увидим в коде |
|---|---|---|
| Абстракция | выделяем главное и прячем детали | класс `BankAccount` описывает банковский счёт, а не весь банк |
| Инкапсуляция | управляем доступом к внутреннему состоянию | `@property`, `_balance` |
| Наследование | переиспользуем общее поведение | `Student` и `Teacher` наследуются от `Person` |
| Полиморфизм | разные объекты имеют общий интерфейс | у разных фигур есть метод `area()` |

Важно: ООП - это не “писать всё через классы”. Это способ организовать код, когда он реально помогает.

Эти четыре идеи не нужно заучивать как отдельные определения. На семинаре мы будем возвращаться к ним через код: где прячем детали, где переиспользуем поведение, где пишем общий интерфейс.

### Классы, объекты и `self`

**Класс** - это описание будущих объектов.  
**Объект** - конкретный экземпляр класса.

Аналогия: класс - чертёж дома, объект - конкретный построенный дом.

По одному классу можно создать сколько угодно объектов. У них будет одинаковый набор возможностей, но свои собственные значения внутри.

In [ ]:
class DummyClass:
    pass


dummy_object = DummyClass()

print(type(dummy_object))
print(isinstance(dummy_object, DummyClass))

<class '__main__.DummyClass'>
True


#### `self`

`self` - ссылка на текущий объект.

Когда мы вызываем:

```python
printer.print_goose()
```

Python фактически делает примерно так:

```python
GoosePrinter.print_goose(printer)
```

Поэтому первым аргументом метода почти всегда пишут `self`.

`self` - не специальное ключевое слово, а обычное имя параметра. Но почти все пишут именно `self`, потому что так код сразу понятен другим Python-разработчикам.

In [ ]:
GOOSE = """
░░░░░▄▀▀▀▄░░░░░░░░
▄███▀░◐░░░▌░░░░░░░
░░░░▌░░░░░▐░░░░░░░
░░░░▐░░░░░▐░░░░░░░
░░░░▌░░░░░▐▄▄░░░░░
░░░░▌░░░░▄▀▒▒▀▀▀▀▄
░░░▐░░░░▐▒▒▒▒▒▒▒▒▀▀▄
░░░▐░░░░▐▄▒▒▒▒▒▒▒▒▒▒▀▄
░░░░▀▄░░░░▀▄▒▒▒▒▒▒▒▒▒▒▀▄
░░░░░░▀▄▄▄▄▄█▄▄▄▄▄▄▄▄▄▄▄▀▄
░░░░░░░░░░░▌▌▌▌░░░░░
░░░░░░░░░░░▌▌░▌▌░░░░░
░░░░░░░░░▄▄▌▌▄▌▌░░░░░
"""


class GoosePrinter:
    def print_goose(self) -> None:
        print(GOOSE)


printer = GoosePrinter()
printer.print_goose()


░░░░░▄▀▀▀▄░░░░░░░░
▄███▀░◐░░░▌░░░░░░░
░░░░▌░░░░░▐░░░░░░░
░░░░▐░░░░░▐░░░░░░░
░░░░▌░░░░░▐▄▄░░░░░
░░░░▌░░░░▄▀▒▒▀▀▀▀▄
░░░▐░░░░▐▒▒▒▒▒▒▒▒▀▀▄
░░░▐░░░░▐▄▒▒▒▒▒▒▒▒▒▒▀▄
░░░░▀▄░░░░▀▄▒▒▒▒▒▒▒▒▒▒▀▄
░░░░░░▀▄▄▄▄▄█▄▄▄▄▄▄▄▄▄▄▄▀▄
░░░░░░░░░░░▌▌▌▌░░░░░
░░░░░░░░░░░▌▌░▌▌░░░░░
░░░░░░░░░▄▄▌▌▄▌▌░░░░░



### `__init__` и атрибуты экземпляра

`__init__` вызывается автоматически при создании объекта. Обычно в нём задают начальное состояние объекта.

Проще думать так: объект уже появился, а `__init__` заполняет его нужными данными. Поэтому почти все строки вида `self.x = ...` и `self.y = ...` живут именно здесь.

In [ ]:
class Point2D:
    #Точка на плоскости.

    def __init__(self, x: int, y: int) -> None:
        self.x = x
        self.y = y

    def print_coords(self) -> None:
        print(f"({self.x}, {self.y})")

    def first_coord(self) -> int:
        return self.x


point = Point2D(10, 20)
point.print_coords()
print(point.first_coord())

(10, 20)
10


### `print` и `return` - не одно и то же

- `print(...)` показывает значение на экране;
- `return ...` возвращает значение из функции/метода, чтобы его можно было использовать дальше.

Если метод только печатает, результат нельзя нормально сложить, сохранить или передать в другую функцию. Поэтому для вычислений почти всегда нужен `return`, а `print` оставляем для показа человеку.

In [ ]:
first = point.first_coord()
print(first + 5)

15


### Атрибуты класса и атрибуты экземпляра

**Атрибут экземпляра** принадлежит конкретному объекту. Обычно создаётся через `self`.

**Атрибут класса** принадлежит самому классу и общий для всех объектов, если его не переопределить.

Хорошая проверка: если значение должно отличаться у разных объектов, кладём его в `self`. Если значение одно на всех, можно оставить его на уровне класса.

In [ ]:
class Student:
    university = "HSE"  # атрибут класса

    def __init__(self, name: str) -> None:
        self.name = name  # атрибут экземпляра


anna = Student("Анна")
ivan = Student("Иван")

print(anna.name, anna.university)
print(ivan.name, ivan.university)

Анна HSE
Иван HSE


#### Типичная ошибка: изменяемый атрибут класса

Список, словарь или множество как атрибут класса часто создают баг: данные становятся общими для всех объектов.

Это особенно неприятно тем, что код выглядит логично, но один студент внезапно получает предметы другого. Причина простая: список был создан один раз на весь класс.

In [ ]:
class BadStudent:
    subjects = []  # ОШИБКА: список общий для всех студентов

    def __init__(self, name: str) -> None:
        self.name = name

    def add_subject(self, subject: str) -> None:
        self.subjects.append(subject)


anna = BadStudent("Анна")
ivan = BadStudent("Иван")

anna.add_subject("Технологии программирования")
print("Предметы Анны:", anna.subjects)
print("Предметы Ивана:", ivan.subjects)

Предметы Анны: ['Технологии программирования']
Предметы Ивана: ['Технологии программирования']


In [ ]:
class GoodStudent:
    def __init__(self, name: str) -> None:
        self.name = name
        self.subjects = []  # правильно: у каждого объекта свой список

    def add_subject(self, subject: str) -> None:
        self.subjects.append(subject)


anna = GoodStudent("Анна")
ivan = GoodStudent("Иван")

anna.add_subject("Технологии  программирования")
print("Предметы Анны:", anna.subjects)
print("Предметы Ивана:", ivan.subjects)

Предметы Анны: ['Технологии  программирования']
Предметы Ивана: []


### Методы и интерфейс объекта

Методы - это действия, которые объект умеет выполнять.

Хороший класс демонстрирует понятный интерфейс: не нужно знать, как всё устроено внутри, достаточно вызвать метод.

Идея интерфейса простая: пользователь класса видит удобные действия вроде `attack()` или `deposit()`, а не копается в том, какие поля и проверки спрятаны внутри.

In [ ]:
class Character:
    def __init__(self, name: str, hp: int, damage: int) -> None:
        self.name = name
        self.hp = hp
        self.damage = damage

    def attack(self, other: "Character") -> None:
        other.hp -= self.damage
        print(f"{self.name} атакует {other.name}. У {other.name} осталось {other.hp} HP")


hero = Character("Hero", 100, 25)
monster = Character("Monster", 80, 15)

hero.attack(monster)
monster.attack(hero)

Hero атакует Monster. У Monster осталось 55 HP
Monster атакует Hero. У Hero осталось 85 HP


### Инкапсуляция и `@property`

В Python нет строгого `private`, как в некоторых других языках.

Но есть несколько соглашений:

- `name` - публичный атрибут;
- `_name` - внутренний атрибут, лучше не трогать снаружи;
- `__name` - name mangling (искажение имён), используется реже.

`@property` позволяет оставить удобный доступ через точку, но добавить проверку или вычисление значения.

Снаружи это выглядит как обычный атрибут: `thermometer.temperature = 30`. Но внутри Python вызывает setter, и мы можем не пустить некорректное значение.

In [ ]:
class Thermometer:
    # Это константа класса: абсолютный ноль один и тот же для всех термометров.
    MINIMAL_TEMPERATURE = -273.15

    def __init__(self, temperature: float) -> None:
        # Здесь мы специально пишем self.temperature, а не self._temperature.
        # Так сразу сработает setter ниже и проверит значение.
        self.temperature = temperature

    @property
    def temperature(self) -> float:
        # Getter вызывается, когда мы читаем thermometer.temperature.
        # Наружу отдаём внутреннее поле _temperature.
        return self._temperature

    @temperature.setter
    def temperature(self, value: float) -> None:
        # Setter вызывается, когда мы пишем thermometer.temperature = value.
        # Здесь удобно держать проверку, чтобы объект не оказался в плохом состоянии.
        if value < self.MINIMAL_TEMPERATURE:
            raise ValueError("Температура не может быть ниже абсолютного нуля")
        # Подчёркивание показывает: это внутреннее поле, напрямую его лучше не трогать.
        self._temperature = value


thermometer = Thermometer(25)
# При чтении temperature сработает getter.
print(thermometer.temperature)

# При записи temperature снова сработает setter и проверит новое значение.
thermometer.temperature = 30
print(thermometer.temperature)

25
30


In [ ]:
try:
    thermometer.temperature = -300
except ValueError as error:
    print("Ожидаемая ошибка:", error)

Ожидаемая ошибка: Температура не может быть ниже абсолютного нуля


### Магические методы

Магические методы начинаются и заканчиваются двойным подчёркиванием: `__method__`.

Обычно мы не вызываем их напрямую. Мы пишем обычный Python-код, а интерпретатор сам обращается к нужному методу.

Они позволяют объектам работать с обычным синтаксисом Python:

- `print(obj)` → `__str__`;
- `repr(obj)` → `__repr__`;
- `a + b` → `__add__`;
- `a == b` → `__eq__`.

#### `__str__` и `__repr__`

- `__str__` - человекочитаемое описание объекта;
- `__repr__` - техническое описание для отладки. В идеале похоже на код, который может создать такой же объект.

В ноутбуке это хорошо видно: `print(p)` использует `__str__`, а просто последняя строка `p` в ячейке обычно показывает `__repr__`.

In [ ]:
class Point2D:
    def __init__(self, x: int, y: int) -> None:
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Point2D({self.x}, {self.y})"

    def __str__(self) -> str:
        return f"Точка с координатами ({self.x}, {self.y})"


p = Point2D(10, 20)
print(p)
print(repr(p))

Точка с координатами (10, 20)
Point2D(10, 20)


#### `__add__` и `__sub__`

Добавим сложение и вычитание точек.

Смысл не в том, что точки обязательно надо складывать в реальном проекте. Важно увидеть механизм: мы сами объясняем Python, что означает оператор `+` для нашего класса.

Если операция не поддерживается, лучше вернуть `NotImplemented`, чтобы Python смог корректно обработать ситуацию.

In [ ]:
class Point2D:
    def __init__(self, x: int, y: int) -> None:
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Point2D({self.x}, {self.y})"

    def __add__(self, other: object) -> Point2D:
        if not isinstance(other, Point2D):
            return NotImplemented
        return Point2D(self.x + other.x, self.y + other.y)

    def __sub__(self, other: object) -> Point2D:
        if not isinstance(other, Point2D):
            return NotImplemented
        return Point2D(self.x - other.x, self.y - other.y)


p1 = Point2D(10, 20)
p2 = Point2D(3, 5)

print(p1 + p2)
print(p1 - p2)

Point2D(13, 25)
Point2D(7, 15)


In [ ]:
try:
    print(p1 + 10)
except TypeError as error:
    print("Ожидаемая ошибка:", error)

Ожидаемая ошибка: unsupported operand type(s) for +: 'Point2D' and 'int'


#### `__eq__`

`__eq__` отвечает за `==` и должен возвращать `bool` или `NotImplemented`.

Без своего `__eq__` объекты обычно сравниваются как разные коробки в памяти. А нам часто нужно сравнивать не коробки, а данные внутри них.

In [ ]:
class Point2D:
    def __init__(self, x: int, y: int) -> None:
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Point2D({self.x}, {self.y})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Point2D):
            return NotImplemented
        return self.x == other.x and self.y == other.y


print(Point2D(1, 2) == Point2D(1, 2))
print(Point2D(1, 2) == Point2D(2, 1))
print(Point2D(1, 2) == "not a point")

True
False
False


### Копирование объектов

Простое присваивание не создаёт новый объект. Оно создаёт ещё одну ссылку на тот же объект.

Поэтому если изменить объект через одну переменную, изменение будет видно и через другую. Это нормально для Python, но часто удивляет на списках и объектах с вложенными данными.

In [ ]:
class Backpack:
    def __init__(self, owner: str, items: list) -> None:
        self.owner = owner
        self.items = items

    def __repr__(self) -> str:
        return f"Backpack(owner={self.owner!r}, items={self.items!r})"


first = Backpack("Анна", ["book", "laptop"])
second = first

second.items.append("charger")

print(first)
print(second)
print(first is second)

Backpack(owner='Анна', items=['book', 'laptop', 'charger'])
Backpack(owner='Анна', items=['book', 'laptop', 'charger'])
True


#### `copy` и `deepcopy`

- `copy` создаёт новый внешний объект, но вложенные объекты остаются общими;
- `deepcopy` рекурсивно копирует и вложенные объекты.

Если внутри объекта лежат только числа и строки, разница может быть незаметна. Если внутри есть список, словарь или другой объект, `deepcopy` часто ведёт себя ближе к тому, чего ожидает новичок.

In [ ]:
original = Backpack("Анна", ["book", "laptop"])
shallow = copy(original)
deep = deepcopy(original)

shallow.items.append("pen")
deep.items.append("notebook")

print("original:", original)
print("shallow: ", shallow)
print("deep:    ", deep)

original: Backpack(owner='Анна', items=['book', 'laptop', 'pen'])
shallow:  Backpack(owner='Анна', items=['book', 'laptop', 'pen'])
deep:     Backpack(owner='Анна', items=['book', 'laptop', 'notebook'])


### Наследование и `super()`

Наследование подходит, когда между классами есть отношение **is-a**:  
`Student` является `Person`, `Teacher` является `Person`.

`super()` позволяет вызвать реализацию родительского класса и не копировать её руками. Это особенно полезно в `__init__`, когда у родителя уже есть общая часть настройки объекта.

Если отношение **has-a** - “имеет внутри себя” - чаще лучше использовать композицию. Например, `Car` имеет `Engine`.

In [ ]:
class Person:
    def __init__(self, name: str, age: int) -> None:
        # Общие поля для всех
        self.name = name
        self.age = age

    def introduce(self) -> str:
        # Общий способ представиться тоже лежит в базовом классе
        return f"Меня зовут {self.name}, мне {self.age} лет"


# Student наследуется от Person: студент - это частный случай человека
class Student(Person):
    def __init__(self, name: str, age: int, group: str) -> None:
        # Не копируем self.name = name и self.age = age
        # Просим родителя Person настроить общую часть объекта
        super().__init__(name, age)
        # А здесь добавляем то, что есть только у студента
        self.group = group

    def introduce(self) -> str:
        # Берём родительское представление и дописываем студенческую часть
        return f"{super().introduce()}, я учусь в группе {self.group}"


student = Student("Анна", 20, "ББИ-2501")
print(student.introduce())

Меня зовут Анна, мне 20 лет, я учусь в группе ББИ-2501


In [ ]:
class Engine:
    def start(self) -> None:
        print("Двигатель запущен")


# Car НЕ наследуется от Engine, потому что машина не является двигателем
# Машина просто содержит двигатель внутри себя: это композиция, отношение has-a
class Car:
    def __init__(self, engine: Engine) -> None:
        # Кладём готовый объект Engine внутрь Car
        self.engine = engine

    def drive(self) -> None:
        # Машина делегирует запуск своему двигателю
        self.engine.start()
        print("Машина едет")


car = Car(Engine())
car.drive()

Двигатель запущен
Машина едет


### Полиморфизм и duck typing

Полиморфизм: один и тот же код работает с разными типами объектов, если у них есть нужный интерфейс.

В Python часто используется duck typing: “если объект крякает как утка, то для нас это утка”. То есть важен не конкретный класс, а наличие нужного метода.

В примере функция make_sound не спрашивает, кто перед ней: собака, кошка или робот. Ей достаточно, чтобы у объекта был метод `speak()`.

In [ ]:
class Dog:
    def speak(self) -> str:
        return "Гав"


class Cat:
    def speak(self) -> str:
        return "Мяу"


class Robot:
    def speak(self) -> str:
        return "Бип-буп"


def make_sound(obj: object) -> None:
    print(obj.speak())


for animal in [Dog(), Cat(), Robot()]:
    make_sound(animal)

Гав
Мяу
Бип-буп


### ABC - абстрактные базовые классы

ABC нужны, когда мы хотим явно зафиксировать интерфейс: какие методы должны быть у наследников.

Например, любая фигура должна уметь считать площадь.

Плюс ABC даёт раннюю ошибку: если наследник забыл реализовать обязательный метод, Python не даст создать такой объект. Это удобнее, чем ловить проблему где-то глубоко в программе.

In [ ]:
class Shape(ABC):
    # abstractmethod означает: у любой конкретной фигуры обязан быть метод area
    # Сам Shape - это не настоящая фигура, а общий шаблон/контракт
    @abstractmethod
    def area(self) -> float:
        # pass здесь нормален: базовый класс только требует метод, но не знает формулу
        pass


class Circle(Shape):
    def __init__(self, radius: float) -> None:
        self.radius = radius

    def area(self) -> float:
        # Круг сам знает свою формулу площади
        return pi * self.radius ** 2


class Rectangle(Shape):
    def __init__(self, width: float, height: float) -> None:
        self.width = width
        self.height = height

    def area(self) -> float:
        # Прямоугольник сам знает свою формулу площади
        return self.width * self.height


# В одном списке лежат разные классы, но у всех есть общий метод area()
shapes = [Circle(3), Rectangle(4, 5)]

for shape in shapes:
    # Нам не важно, круг это или прямоугольник: просто вызываем общий интерфейс
    print(shape.area())

28.274333882308138
20


In [ ]:
# Shape нельзя создать напрямую: это абстрактная идея фигуры, а не конкретная фигура
try:
    Shape()
except TypeError as error:
    print("Ожидаемая ошибка:", error)

Ожидаемая ошибка: Can't instantiate abstract class Shape without an implementation for abstract method 'area'


#### Если наследник забыл метод

Иногда наследник реализует часть интерфейса, но один обязательный метод случайно упускает. Для ABC это всё ещё незавершённый класс, поэтому создать объект не получится.

In [ ]:
class Exporter(ABC):
    # Наследники должны уметь экспортировать данные...
    @abstractmethod
    def export(self) -> str:
        pass

    # ...и сообщать расширение файла.
    @abstractmethod
    def file_extension(self) -> str:
        pass


class JsonExporter(Exporter):
    def export(self) -> str:
        return "{}"

    # Метод file_extension забыли реализовать, поэтому класс ещё не готов.


try:
    JsonExporter()
except TypeError as error:
    print("Ожидаемая ошибка:", error)

Ожидаемая ошибка: Can't instantiate abstract class JsonExporter without an implementation for abstract method 'file_extension'


### `@staticmethod` и `@classmethod`

#### `@staticmethod`

Статический метод лежит внутри класса, но не получает ни `self`, ни `cls`.

Используется для вспомогательных функций, которые логически относятся к классу.

Если функция вообще не связана с классом по смыслу, её лучше оставить обычной функцией. `staticmethod` нужен именно когда хочется держать маленький помощник рядом с классом.

In [ ]:
class MathUtils:
    # Метод не использует данные конкретного объекта, поэтому self не нужен
    # Он просто лежит рядом с классом, потому что по смыслу относится к математике
    @staticmethod
    def is_positive(number: int) -> bool:
        return number > 0


# Объект MathUtils создавать не нужно: вызываем метод прямо от класса
print(MathUtils.is_positive(10))
print(MathUtils.is_positive(-5))

True
False


#### `@classmethod`

Метод класса получает первым аргументом `cls` - сам класс.

Часто используется как альтернативный конструктор.

Слово `cls` здесь важно: метод создаёт объект того класса, на котором его вызвали. Это делает код дружелюбнее к наследованию.

In [ ]:
class Temperature:
    def __init__(self, celsius: float) -> None:
        # Основной способ хранения температуры внутри класса - градусы Цельсия
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, fahrenheit: float) -> Temperature:
        # Пользователь дал температуру в Фаренгейтах
        # Сначала переводим её в Цельсии...
        celsius = (fahrenheit - 32) * 5 / 9
        # ...а потом создаём объект нашего класса.
        # cls здесь обычно означает Temperature, но при наследовании может быть и подкласс
        return cls(celsius)

    def __repr__(self) -> str:
        return f"Temperature({self.celsius})"


# Это альтернативный конструктор: создаём объект не напрямую через Celsius,
# а через удобный метод для Fahrenheit
temperature = Temperature.from_fahrenheit(100)
print(temperature)

Temperature(37.77777777777778)


### `dataclass`

`dataclass` удобен, когда класс в основном хранит данные.

Он автоматически создаёт часть служебных методов, например `__init__`, `__repr__`, `__eq__`.

То есть мы описываем поля, а Python дописывает рутинный код за нас. Но если в классе много сложного поведения и проверок, обычный класс всё ещё может быть понятнее.

In [ ]:
# просто перечисляем поля объекта.
@dataclass
class User:
    # Обязательные поля: их нужно передать при создании User
    name: str
    age: int
    # Поле со значением по умолчанию: его можно не передавать
    city: str = "Москва"


# __init__ мы не писали, но он уже есть благодаря @dataclass
user_1 = User("Анна", 20)
user_2 = User("Анна", 20)

# __repr__ тоже сгенерирован: объект красиво печатается
print(user_1)
# __eq__ тоже сгенерирован: объекты сравниваются по значениям полей
print(user_1 == user_2)

User(name='Анна', age=20, city='Москва')
True


#### Изменяемые значения в `dataclass`

Для списков и словарей используем `field(default_factory=...)`, а не `=[]`.

`default_factory` создаёт новый список для каждого объекта. Это ровно та же проблема с общими изменяемыми значениями, которую мы уже видели у атрибутов класса.

In [ ]:
@dataclass
class Course:
    title: str
    # Нельзя писать students: list = [], иначе один список стал бы общим для всех курсов
    # default_factory=list создаёт новый пустой список для каждого куурса
    students: list = field(default_factory=list)

    def add_student(self, name: str) -> None:
        # Добавляем студента именно в список этого конкретного курса
        self.students.append(name)


tp_course = Course("Технологии программирования")
math_course = Course("Математический анализ")

tp_course.add_student("Анна")

# У ТП студент появился
print(tp_course)
# У матанализа список остался пустым: значит списки разные, всё хорошо
print(math_course)

Course(title='Технологии программирования', students=['Анна'])
Course(title='Математический анализ', students=[])


### Задачи для самостоятельного решения

#### Задание 1. `BankAccount`

Реализуйте класс `BankAccount`.

Требования:

1. У счёта есть владелец и баланс.
2. Баланс нельзя сделать отрицательным.
3. Метод `deposit(amount)` пополняет счёт.
4. Метод `withdraw(amount)` снимает деньги.
5. Если денег не хватает - выбрасывается `ValueError`.
6. `print(account)` должен выводить понятное описание счёта.

*Подсказка*: баланс лучше хранить как внутреннее поле, а наружу отдавать через `@property`. Так мы не дадим случайно записать в счёт отрицательное значение.

In [ ]:
class BankAccountTask:
    # Ваш код здесь
    pass

#### Задание 2. `Vector2D`

Реализуйте класс `Vector2D`.

Требования:

1. Поля `x`, `y`.
2. Красивый `__repr__`.
3. Сложение векторов через `+`.
4. Вычитание векторов через `-`.
5. Проверка равенства через `==`.
6. Метод `length()` возвращает длину вектора.

Здесь тренируем магические методы: после их реализации объект начинает вести себя почти как встроенный тип. Пользователь пишет `v1 + v2`, а не вызывает отдельную функцию сложения.

In [ ]:
class Vector2DTask:
    # Ваш код здесь
    pass

#### Задание 3. Фигуры

Реализуйте систему классов:

1. Абстрактный класс `ShapeTask` с методом `area()`.
2. Класс `CircleTask`.
3. Класс `RectangleTask`.
4. Функцию `total_area(shapes)`, которая считает суммарную площадь.

Главная цель задания - увидеть полиморфизм: функция `total_area` не должна знать, какие конкретно фигуры лежат в списке.

Пусть каждый класс сам отвечает за свою формулу площади. Тогда общая функция просто вызывает `area()` и не разрастается условиями `if circle`, `if rectangle`.

In [ ]:
class ShapeTask:
    # Ваш код здесь
    pass

## Раздел 2. Тестирование

В этом разделе мы с вами научимся тестировать код:

- Рассмотрим существующие фреймворки для тестирования в Python
- На примере `pytest` разберемся, как работает функционал тестирования
- Познакомимся с концепцией `TDD`, понятиями **ката**, **параметризация**, **фикстуры**
- Поработаем с исключениями при тестировании
- Конечно же, попрактикуемся!

### Сравнение тестовых фреймворков Python

| Фреймворк  | Плюсы                                                                 | Минусы                                                                 |
|------------|-----------------------------------------------------------------------|------------------------------------------------------------------------|
| **unittest** | • Входит в стандартную библиотеку<br>• JUnit-стиль (знаком многим)<br>• Поддержка фикстур (setUp/tearDown) | • Много boilerplate-кода<br>• Менее читаемые ассерты<br>• Синтаксис сложнее, чем у современных аналогов |
| **pytest**   | • Минимальный boilerplate<br>• Читаемые ассерты (`assert x == y`)<br>• Параметризация тестов<br>• Плагины и богатая экосистема<br>• Поддержка unittest-тестов | • Требует установки (не в стандартной библиотеке)<br>• Синтаксис фикстур может быть сложным для новичков |
| **doctest**  | • Встроен в стандартную библиотеку<br>• Тесты как документация<br>• Нет лишнего кода | • Не подходит для сложных тестов<br>• Сложность отладки<br>• Может нарушать читаемость документации |
| **nose2**    | • Совместимость с unittest<br>• Упрощенный запуск тестов<br>• Плагины | • Развитие замедлилось после появления pytest<br>• Меньше возможностей, чем у pytest |

**Рекомендации**:
- **pytest** — лучший выбор для большинства проектов

### Именование тестов

Как вообще утилита **pytest** находит тесты среди прочих файлов?

1. Находит все файлы, которые оканчиваются на *.py
2. Оставляет только файлы вида `test_*.py`, `*_test.py`
3. Внутри файлов:
    * находит все функции с префиксом `test`;
    * находит все методы с префиксом `test` внутри класса с прекфиксом `Test`.

При необходимости можно обратиться к [конвенции](https://docs.pytest.org/en/stable/explanation/goodpractices.html#test-discovery)

### TDD

**TDD** - Test Driven Development: сперва пишем тест для необходимой логики, затем пишем код для этой логики, основываясь на нашем тесте.

In [ ]:
%pip install pytest ipytest --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 16.5 MB/s eta 0:00:00


In [ ]:
import pytest
import ipytest

ipytest.autoconfig()

In [ ]:
%%ipytest


def test_good_math():
    assert 2 + 2 == 4


def test_bad_math():
    assert 2 + 2 == 5

.F                                                                                           [100%]
============================================= FAILURES =============================================
__________________________________________ test_bad_math ___________________________________________

    def test_bad_math():
>       assert 2 + 2 == 5
E       assert (2 + 2) == 5

/tmp/ipykernel_699/3932585718.py:6: AssertionError
===================================== short test summary info ======================================
FAILED t_a53e511ba8ea4f59aaa6db49e4132afa.py::test_bad_math - assert (2 + 2) == 5
1 failed, 1 passed in 0.32s


`pytest` выводит отчет, в котором можно посмотреть сколько у нас всего тестов, какие из них упали и по какой причине.

Три правила TDD:

1. Продакшн-код можно писать только для починки падающего теста.

2. В тесте нужно писать ровно столько кода, сколько необходимо, чтобы он упал. Ошибки компиляции считаются падениями теста.

3. В продакшн можно написать ровно столько кода, сколько требуется для починки одного падающего теста.

Получается следуйющий пайплайн: пишем падающий тест, пишем код, чтобы тест не падал, рефакторим код так, чтобы тесты не падали. Повторяем до сходимости.

1. [Test Driven Development: By Example 1st Edition](https://www.amazon.com/Test-Driven-Development-Kent-Beck/dp/0321146530)

2. [On Growing Object Oriented Software, Guided by Tests](https://www.amazon.com/Growing-Object-Oriented-Software-Guided-Tests/dp/0321503627)

### Ката

Каты — упражнения по программированию, помогающие отточить навыки путем многократного повторнения. Концепция взята из японских боевых искусств. Подробнее про них можно почитать в книжке [The Pragmatic Programmer](https://pragprog.com/titles/tpp20/the-pragmatic-programmer-20th-anniversary-edition/)

#### Ката Greeter

Эту кату надо выполнять строго по пунктам, не заглядывая вперёд.

* Создайте класс `Greeter`, у которого есть метод `greet` принимающий на вход имя и возвращающий "Hello <имя>".

Давайте сперва напишем тест, в котором проверяем конструктор класса:

In [ ]:
%%ipytest


def test_greeter():
    Greeter()

F                                                                                            [100%]
============================================= FAILURES =============================================
___________________________________________ test_greeter ___________________________________________

    def test_greeter():
>       Greeter()
        ^^^^^^^
E       NameError: name 'Greeter' is not defined

/tmp/ipykernel_699/2715718898.py:2: NameError
===================================== short test summary info ======================================
FAILED t_a53e511ba8ea4f59aaa6db49e4132afa.py::test_greeter - NameError: name 'Greeter' is not defined
1 failed in 0.01s


Теперь нам нужно починить тест минимальным количеством кода:

In [ ]:
class Greeter: ...

Далее сделаем еще одну итерацию написания теста:

In [ ]:
%%ipytest


def test_greeter_2():
    Greeter().greet("Daniel")

F                                                                                            [100%]
============================================= FAILURES =============================================
__________________________________________ test_greeter_2 __________________________________________

    def test_greeter_2():
>       Greeter().greet("Daniel")
        ^^^^^^^^^^^^^^^
E       AttributeError: 'Greeter' object has no attribute 'greet'

/tmp/ipykernel_699/3222435460.py:2: AttributeError
===================================== short test summary info ======================================
FAILED t_a53e511ba8ea4f59aaa6db49e4132afa.py::test_greeter_2 - AttributeError: 'Greeter' object has no attribute 'greet'
1 failed in 0.01s


In [ ]:
class Greeter:  # noqa: F811
    def greet(self, name: str) -> None:
        return f"Hello, {name}!"

В конце концов напишем финальный тест, который что-то проверяет. Тут нам поможет знакомый нам `assert`:

In [ ]:
%%ipytest


def test_greeter_3():
    name = "Daniel"
    assert Greeter().greet(name=name) == f"Hello, {name}!"

.                                                                                            [100%]
1 passed in 0.01s


### Параметризация

Давайте теперь напишем несколько тестов, проверяющих наш метод:

In [ ]:
%%ipytest


def test_greeter_name():
    name = "Daniel"
    assert Greeter().greet(name=name) == f"Hello, {name}!"


def test_greeter_empty():
    name = ""
    assert Greeter().greet(name=name) == f"Hello, {name}!"

..                                                                                           [100%]
2 passed in 0.01s


Чем больше кейсов, которые мы хотим проверить, тем больше у нас тестирующих функций - это не очень удобно. Давайте посмотрим как можно делать несколько разных кейсов в одном тесте. С этим нам поможет *параметризация*:

In [ ]:
%%ipytest
test_cases = [
    ("Daniel", "Hello, Daniel!"),
    ("", "Hello, !"),
]


@pytest.mark.parametrize("name, greeting", test_cases)
def test_greeter_parametrized(name, greeting):
    assert Greeter().greet(name) == greeting

..                                                                                           [100%]
2 passed in 0.01s


Следующий пункт нашей каты: метод `greet` должен обрезать пробельные символы в по краям имени.

In [ ]:
%%ipytest
test_cases = [("Daniel", "Hello, Daniel!"), ("", "Hello, !"), ("\n\t\n\t", "Hello, !")]


@pytest.mark.parametrize("name, greeting", test_cases)
def test_greeter_with_trim(name, greeting):
    assert Greeter().greet(name) == greeting

..F                                                                                          [100%]
============================================= FAILURES =============================================
____________________________ test_greeter_with_trim[\n\t\n\t-Hello, !] _____________________________

name = '\n\t\n\t', greeting = 'Hello, !'

    @pytest.mark.parametrize("name, greeting", test_cases)
    def test_greeter_with_trim(name, greeting):
>       assert Greeter().greet(name) == greeting
E       AssertionError: assert 'Hello, \n\t\n\t!' == 'Hello, !'
E         
E         - Hello, !
E         ?        ^
E         + Hello, 
E         ?        ^
E         + 	
E         + 	!

/tmp/ipykernel_699/3799989716.py:6: AssertionError
===================================== short test summary info ======================================
FAILED t_a53e511ba8ea4f59aaa6db49e4132afa.py::test_greeter_with_trim[\n\t\n\t-Hello, !] - AssertionError: assert 'Hello, \n\t\n\t!' == 'Hello, !'
1 failed, 2 p

Пора опять чинить код, чтобы он проходил наш тест:

In [ ]:
class Greeter:  # noqa: F811
    def greet(self, name: str) -> None:
        return f"Hello, {name.strip()}!"

In [ ]:
%%ipytest
test_cases = [("Daniel", "Hello, Daniel!"), ("", "Hello, !"), ("\n\t\n\t", "Hello, !")]


@pytest.mark.parametrize("name, greeting", test_cases)
def test_greeter_with_trim(name, greeting):  # noqa: F811
    assert Greeter().greet(name) == greeting

...                                                                                          [100%]
3 passed in 0.02s


### Фикстуры

Фикстуры - очень сильный механизм, помогающий создать общий контекст для тестов.

In [ ]:
%%ipytest


class Fruit:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name


@pytest.fixture
def my_fruit():
    return Fruit("apple")


@pytest.fixture
def fruit_basket(my_fruit):
    return [Fruit("banana"), my_fruit]


def test_my_fruit_in_basket(my_fruit, fruit_basket):
    assert my_fruit in fruit_basket

.                                                                                            [100%]
1 passed in 0.01s


В нашем случае мы для каждого тест кейса создаем объект класса `Greeter`, что нам не особо нужно. Давайте попробуем решить эту проблему с помощью фикстур:

In [ ]:
%%ipytest

from typing import Iterator

test_cases = [("Daniel", "Hello, Daniel!"), ("", "Hello, !"), ("\n\t\n\t", "Hello, !")]


@pytest.fixture(scope="module")
def greeter() -> Iterator[Greeter]:
    yield Greeter()


@pytest.mark.parametrize("name, greeting", test_cases)
def test_greeter_with_trim(greeter: Greeter, name: str, greeting: str):  # noqa: F811
    assert greeter.greet(name) == greeting

...                                                                                          [100%]
3 passed in 0.01s


Фикстуры могут иметь разный скоуп - как для целого модуля тестов, так и для отдельного теста. Фикстуры часто используются как контекстные менеджеры: ведь удобно в тесте подключиться к чему-то, а после теста закрыть это дело. Простой пример из жизни - подключение к базе данных в тесте, оно должно быть закрыто после выполнения теста.

In [ ]:
%%ipytest -s


class DBConnection:
    pass


class TestDB:
    def init_db(self):
        print("init db")

    def get_connection(self) -> DBConnection:
        return DBConnection()

    def shutdown(self):
        print("close db")


@pytest.fixture(scope="module")
def db_connection() -> Iterator[DBConnection]:
    db = TestDB()
    db.init_db()
    try:
        yield db.get_connection()
    finally:
        db.shutdown()


def test_db_1(db_connection: DBConnection):
    assert db_connection


def test_db_2(db_connection: DBConnection):
    assert db_connection

init db
..close db

2 passed in 0.01s


Тест не ограничивается одной фикстурой в аргументе: можно прописывать сколько угодно фикстур для тестов.

In [ ]:
%%ipytest

import pytest


@pytest.fixture
def first_entry():
    return "a"


@pytest.fixture
def second_entry():
    return 2


@pytest.fixture
def order(first_entry, second_entry):
    return [first_entry, second_entry]


@pytest.fixture
def expected_list():
    return ["a", 2, 3.0]


def test_string(order, expected_list):
    order.append(3.0)

    assert order == expected_list

.                                                                                            [100%]
1 passed in 0.01s


Фикстура может автоматически использоваться для каждого теста.

In [ ]:
%%ipytest

import pytest


@pytest.fixture
def first_entry():  # noqa: F811
    return "a"


@pytest.fixture
def order(first_entry):  # noqa: F811
    return []


@pytest.fixture(autouse=True)
def append_first(order, first_entry):
    return order.append(first_entry)


def test_string_only(order, first_entry):
    assert order == [first_entry]


def test_string_and_int(order, first_entry):
    order.append(2)
    assert order == [first_entry, 2]

..                                                                                           [100%]
2 passed in 0.01s


### Тестирование исключений

Следующий пункт нашей каты: метод `greet` должен возвращать исключение если передана пустая строка (или строка, состоящая только из пробельных символов).

Для тестирования исключений есть специальный контекстный менеджер в модуле `pytest`.

In [ ]:
%%ipytest

from typing import Iterator

test_cases = [("Daniel", "Hello, Daniel!"), ("", "Hello, !"), ("\n\t\n\t", "Hello, !")]


@pytest.fixture(scope="module")
def greeter() -> Iterator[Greeter]:  # noqa: F811
    yield Greeter()


@pytest.mark.parametrize("name, greeting", test_cases)
def test_greeter_with_trim(greeter: Greeter, name: str, greeting: str):  # noqa: F811
    with pytest.raises(ValueError):
        greeter.greet("")

FFF                                                                                          [100%]
============================================= FAILURES =============================================
__________________________ test_greeter_with_trim[Daniel-Hello, Daniel!] ___________________________

greeter = <__main__.Greeter object at 0x7ebad138db20>, name = 'Daniel', greeting = 'Hello, Daniel!'

    @pytest.mark.parametrize("name, greeting", test_cases)
    def test_greeter_with_trim(greeter: Greeter, name: str, greeting: str):  # noqa: F811
>       with pytest.raises(ValueError):
             ^^^^^^^^^^^^^^^^^^^^^^^^^
E       Failed: DID NOT RAISE <class 'ValueError'>

/tmp/ipykernel_699/3299238826.py:13: Failed
________________________________ test_greeter_with_trim[-Hello, !] _________________________________

greeter = <__main__.Greeter object at 0x7ebad138db20>, name = '', greeting = 'Hello, !'

    @pytest.mark.parametrize("name, greeting", test_cases)
    def test_greeter_w

Исправим наш класс, чтобы он проходил тесты:

In [ ]:
class Greeter:  # noqa: F811
    def greet(self, name: str) -> None:
        name = name.strip()

        if name == "":
            raise ValueError("Empty name")

        return f"Hello, {name.strip()}!"

In [ ]:
%%ipytest

from typing import Iterator

test_cases = [
    ("Daniel", "Hello, Daniel!", False),
    ("", "Hello, !", True),
    ("\n\t\n\t", "Hello, !", True),
]


@pytest.fixture(scope="module")
def greeter() -> Iterator[Greeter]:  # noqa: F811
    yield Greeter()


@pytest.mark.parametrize("name, greeting, want_error", test_cases)
def test_greeter_with_trim(  # noqa: F811
    greeter: Greeter, name: str, greeting: str, want_error: bool
):
    if want_error:
        with pytest.raises(ValueError):
            greeter.greet(name)
    else:
        greeter.greet(name)

...                                                                                          [100%]
3 passed in 0.02s


## Сравнение float

In [ ]:
%%ipytest


def test_float():
    assert 0.1 + 0.2 == 0.3

F                                                                                            [100%]
============================================= FAILURES =============================================
____________________________________________ test_float ____________________________________________

    def test_float():
>       assert 0.1 + 0.2 == 0.3
E       assert (0.1 + 0.2) == 0.3

/tmp/ipykernel_699/2584694480.py:2: AssertionError
===================================== short test summary info ======================================
FAILED t_a53e511ba8ea4f59aaa6db49e4132afa.py::test_float - assert (0.1 + 0.2) == 0.3
1 failed in 0.01s


In [ ]:
%%ipytest


def test_float():  # noqa: F811
    assert 0.1 + 0.2 == pytest.approx(0.3)

.                                                                                            [100%]
1 passed in 0.01s


In [ ]:
%%ipytest


def test_float():  # noqa: F811
    assert [0.1 + 0.2, 0.5] == pytest.approx([0.3, 0.5])

.                                                                                            [100%]
1 passed in 0.01s


## Попрактикуемся

Нам нужно покрыть следующий модуль тестами:

In [ ]:
class MailAdminClient:
    def create_user(self) -> "MailUser":
        return MailUser()

    def delete_user(self, user: "MailUser") -> None:
        pass


class MailUser:
    def __init__(self):
        self.inbox = []

    def send_email(self, email: "Email", other: "MailUser") -> None:
        other.inbox.append(email)

    def clear_mailbox(self) -> None:
        self.inbox.clear()


class Email:
    def __init__(self, subject: str, body: str):
        self.subject = subject
        self.body = body

In [ ]:
%%ipytest

import pytest
from typing import Iterator


@pytest.fixture
def mail_admin():
    return MailAdminClient()


@pytest.fixture
def sending_user(mail_admin) -> Iterator[MailUser]:
    user = mail_admin.create_user()
    yield user
    mail_admin.delete_user(user)


@pytest.fixture
def receiving_user(mail_admin) -> Iterator[MailUser]:
    user = mail_admin.create_user()
    yield user
    user.clear_mailbox()
    mail_admin.delete_user(user)


def test_email_received(sending_user, receiving_user):
    email = Email(subject="Hey!", body="How's it going?")
    sending_user.send_email(email, receiving_user)
    assert email in receiving_user.inbox

.                                                                                            [100%]
1 passed in 0.01s


Посмотрим еще на пример интеграционного тестирования некоторого метода:

In [ ]:
%%writefile weather_func.py
import requests

def get_weather_wttr(city="moscow"):
    url = f"https://wttr.in/{city}?format=j1"
    response = requests.get(url, timeout=10)
    return response.json()

Writing weather_func.py


In [ ]:
%%ipytest

from weather_func import get_weather_wttr

@pytest.mark.integration  # Помечаем как интеграционный тест
def test_real_api_connection():
    result = get_weather_wttr("Moscow")

    assert result is not None
    assert "current_condition" in result
    assert len(result["current_condition"]) > 0

    current = result["current_condition"][0]
    assert "temp_C" in current
    assert "weatherDesc" in current

.                                                                                            [100%]
1 passed in 1.19s
